# Dashboard — Warehouse & Retail Sales

**Project:** Warehouse & Retail Sales ML Pipeline  
**Author:** Santiago López Blanco  
**Date:** May 2026  
**Sources:** `main.default.gold_business_metrics` · `main.default.gold_ml_features`

## Goal
Translate the data processed across the Bronze → Silver → Gold pipeline into clear business visualizations. This notebook answers four business questions:

1. **Which product categories and suppliers drive the most volume?**
2. **How do sales trends behave over time by category?**
3. **How accurate is the ML model across different segments?**
4. **What does the demand forecast look like for 2025?**

## Notebook Structure

| Section | What it does |
|---------|-------------|
| **Setup & Data Load** | Loads Gold tables into pandas for visualization |
| **Sales Overview** | Total sales by category and top suppliers |
| **Trends** | Monthly sales evolution by item type |
| **Model Performance** | MAE and R² breakdown by segment |
| **Forecast** | Simulated 2025 demand forecast using the registered LightGBM model |

## Setup & Data Load

Loads all required libraries and reads data from four Gold tables in Unity Catalog.
Aggregations run in Spark before transferring to pandas — only the final results move to the driver.

| Data | Source | Rows transferred | Strategy |
|------|--------|-----------------|----------|
| `sales_by_type` | `gold_business_metrics` | 6 | Aggregated in Spark |
| `sales_by_supplier` | `gold_business_metrics` | 15 | Aggregated in Spark |
| `trends_df` | `gold_business_metrics` | 144 | Aggregated in Spark |
| `ml_df` | `gold_ml_features` | 8,219 | Full load — needed for forecast |
| `perf_type_df` | `gold_model_performance_by_type` | 6 | Full load — already aggregated |
| `perf_tier_df` | `gold_model_performance_by_tier` | 3 | Full load — already aggregated |

In [0]:
# SETUP & DATA LOAD
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Load only what each section needs — aggregations run in Spark before moving to pandas
# ml_df and perf tables are small enough to load fully

# Full load only for tables that need row-level access in forecast
ml_df       = spark.table("main.default.gold_ml_features").toPandas()
perf_type_df = spark.table("main.default.gold_model_performance_by_type").toPandas()
perf_tier_df = spark.table("main.default.gold_model_performance_by_tier").toPandas()

# Aggregated loads — Spark computes before transferring to pandas
sales_by_type = spark.sql("""
    SELECT item_type, SUM(total_sales) AS total_sales
    FROM main.default.gold_business_metrics
    GROUP BY item_type
    ORDER BY total_sales ASC
""").toPandas()

sales_by_supplier = spark.sql("""
    SELECT supplier, SUM(total_sales) AS total_sales
    FROM main.default.gold_business_metrics
    GROUP BY supplier
    ORDER BY total_sales DESC
    LIMIT 15
""").toPandas()

trends_df = spark.sql("""
    SELECT year, month, item_type, SUM(total_sales) AS total_sales
    FROM main.default.gold_business_metrics
    GROUP BY year, month, item_type
    ORDER BY year, month
""").toPandas()

# Summary
print(f"✓ Data loaded")
print(f"  sales_by_type     : {len(sales_by_type)} rows")
print(f"  sales_by_supplier : {len(sales_by_supplier)} rows")
print(f"  trends_df         : {len(trends_df)} rows")
print(f"  ml_df             : {len(ml_df):,} rows")
print(f"  perf_type_df      : {len(perf_type_df)} rows")
print(f"  perf_tier_df      : {len(perf_tier_df)} rows")

## Sales Overview

Total sales volume by product category and market concentration by supplier.

**Chart 1 — Total Sales by Item Type**
BEER dominates the market with 7.67M units — more than 3x the second category.
WINE follows at ~2.5M and LIQUOR at ~1.7M. KEGS, NON-ALCOHOL and STR_SUPPLIES
are negligible in volume terms — nearly invisible at this scale.

**Chart 2 — Top 15 Suppliers by Total Sales**
Crown Imports leads with ~1.8M units, followed closely by Anheuser Busch and
Miller Brewing at ~1.55M each. The top 3 suppliers together control the majority
of market volume — confirming the 41.2% concentration identified in the EDA.
From position 4 onwards volume drops sharply — Heineken USA at ~0.9M already
represents less than half of Crown Imports. The remaining 11 suppliers in the
top 15 each contribute less than 0.5M units.

**Key finding:** The market is highly concentrated in one product category (BEER)
and three suppliers (Crown Imports, Anheuser Busch, Miller Brewing). Any demand
shock affecting BEER or these three suppliers has outsized impact on total
warehouse volume.

In [0]:
# SALES OVERVIEW

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1 — Total sales by item type
# Data already aggregated and sorted in Spark
axes[0].barh(sales_by_type["item_type"], sales_by_type["total_sales"], color="#1D9E75")
axes[0].set_title("Total Sales by Item Type", fontsize=13)
axes[0].set_xlabel("Total Sales (units)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
axes[0].spines[["top", "right"]].set_visible(False)

# Chart 2 — Top 15 suppliers by total sales
# Reverse for horizontal bar (largest at top)
sales_by_supplier_plot = sales_by_supplier.sort_values("total_sales", ascending=True)
axes[1].barh(sales_by_supplier_plot["supplier"], sales_by_supplier_plot["total_sales"], color="#2D6FA4")
axes[1].set_title("Top 15 Suppliers by Total Sales", fontsize=13)
axes[1].set_xlabel("Total Sales (units)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## Sales Trends

Monthly sales evolution by item type from 2017 to 2020.

**Chart — Monthly Sales by Item Type (2017–2020)**
BEER dominates so heavily that all other categories appear flat by comparison. WINE is the second most visible line at ~100K units monthly, followed by LIQUOR at ~60-80K. KEGS, NON-ALCOHOL and STR_SUPPLIES are nearly invisible at this scale.

**BEER seasonality:** Clear peaks in summer months (May–September) every year — consistent with outdoor consumption patterns. The highest peak occurs in mid-2019 at ~400K units.

**COVID-19 signal:** In early 2020 BEER shows a sharp drop followed by an unusual spike — anticipatory buying before lockdowns, then a collapse as bars and restaurants closed. This is the exact demand shock that caused the largest model errors in March 2020.

**WIN

In [0]:
# SALES TRENDS

# Create date column for x axis
trends_df["date"] = pd.to_datetime(
    trends_df["year"].astype(str) + "-" + trends_df["month"].astype(str) + "-01"
)

fig, ax = plt.subplots(figsize=(14, 5))

colors = {
    "BEER"        : "#1D9E75",
    "WINE"        : "#2D6FA4",
    "LIQUOR"      : "#D85A30",
    "KEGS"        : "#8B5CF6",
    "NON-ALCOHOL" : "#F59E0B",
    "STR_SUPPLIES": "#6B7280"
}

for item_type, group in trends_df.groupby("item_type"):
    group = group.sort_values("date")
    ax.plot(group["date"], group["total_sales"],
            label=item_type,
            color=colors.get(item_type, "#999"),
            linewidth=1.8)

ax.set_title("Monthly Sales by Item Type (2017–2020)", fontsize=13)
ax.set_xlabel("Date")
ax.set_ylabel("Total Sales (units)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", frameon=False)

plt.tight_layout()
plt.show()

## Model Performance by Segment

LightGBM test performance broken down by product category and supplier tier.
MAE measures average dollar error — lower is better. R² measures variance explained — higher is better.
A negative R² means the model performs worse than simply predicting the average.

**By product type:**
BEER has the highest MAE ($909.29) but also the strongest R² (0.9687) — the large dollar error is driven by the COVID-19 demand shock in March 2020, not by systematic model failure. The underlying sales pattern is well captured.
LIQUOR (0.9525) and WINE (0.8987) are production-ready segments.
KEGS is the only segment with negative R² (-0.0257) — the model performs worse than predicting the average. Keg demand is event-driven and cannot be captured by lag features alone.
NON-ALCOHOL explains only 50.27% of variance — insufficient for reliable forecasting.

**By supplier tier:**
MAE alone is misleading across tiers because sales volumes differ by orders of magnitude.
`rest` has the lowest dollar error ($111.75) simply because their sales volumes are small.
`top3` has the highest dollar error ($3,158.06) but the strongest R² (0.9732) — the pattern is well captured despite large absolute errors.
`top15` has the weakest R² (0.8700) — moderate volume combined with high variability makes this the hardest tier to forecast.

**Key finding:** R² is the correct metric for cross-segment comparison. By that measure, top3 suppliers and BEER are the best predicted segments in the dataset.

In [0]:
# MODEL PERFORMANCE BY SEGMENT

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sort by MAE for consistent ordering
perf_type_sorted = perf_type_df.sort_values("mae")
perf_tier_sorted  = perf_tier_df.sort_values("mae")

# Chart 1 — MAE by item_type
axes[0, 0].barh(perf_type_sorted["item_type"], perf_type_sorted["mae"], color="#2D6FA4")
axes[0, 0].set_title("Test MAE by Item Type (LightGBM)", fontsize=12)
axes[0, 0].set_xlabel("MAE (units)")
axes[0, 0].spines[["top", "right"]].set_visible(False)
for i, v in enumerate(perf_type_sorted["mae"]):
    axes[0, 0].text(v + 5, i, f"{v:,.2f}", va="center", fontsize=9)

# Chart 2 — R² by item_type
bar_colors = ["#E24B4A" if v < 0 else "#1D9E75" for v in perf_type_sorted["r2"]]
axes[0, 1].barh(perf_type_sorted["item_type"], perf_type_sorted["r2"], color=bar_colors)
axes[0, 1].set_title("Test R² by Item Type (LightGBM)", fontsize=12)
axes[0, 1].set_xlabel("R²")
axes[0, 1].axvline(x=0, color="#333", linewidth=0.8, linestyle="--")
axes[0, 1].spines[["top", "right"]].set_visible(False)
for i, v in enumerate(perf_type_sorted["r2"]):
    offset = 0.01 if v >= 0 else -0.06
    axes[0, 1].text(v + offset, i, f"{v:.4f}", va="center", fontsize=9)

# Chart 3 — MAE by supplier_tier
axes[1, 0].barh(perf_tier_sorted["supplier_tier"], perf_tier_sorted["mae"], color="#2D6FA4")
axes[1, 0].set_title("Test MAE by Supplier Tier (LightGBM)", fontsize=12)
axes[1, 0].set_xlabel("MAE (units)")
axes[1, 0].spines[["top", "right"]].set_visible(False)
axes[1, 0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for i, v in enumerate(perf_tier_sorted["mae"]):
    axes[1, 0].text(v + 10, i, f"{v:,.2f}", va="center", fontsize=9)

# Chart 4 — R² by supplier_tier
axes[1, 1].barh(perf_tier_sorted["supplier_tier"], perf_tier_sorted["r2"], color="#1D9E75")
axes[1, 1].set_title("Test R² by Supplier Tier (LightGBM)", fontsize=12)
axes[1, 1].set_xlabel("R²")
axes[1, 1].spines[["top", "right"]].set_visible(False)
for i, v in enumerate(perf_tier_sorted["r2"]):
    axes[1, 1].text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=9)

plt.suptitle("LightGBM — Model Performance by Segment", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Demand Forecast — 2025

LightGBM predictions for January–December 2025 based on the last known feature values
from the Gold table. Model and encoders are loaded dynamically from the latest registered
version in MLflow — no hardcoded version numbers. Forecast covers the top 3 suppliers
calculated automatically from the data across all product categories.

**Chart 1 — Forecast by Item Type**
BEER dominates so heavily that all other categories appear flat at this scale —
consistent with the EDA finding that BEER represents 62.8% of total market volume.
BEER shows a clear seasonal pattern: rising from ~175K in January to a peak of ~225K
in April–June, then declining through Q3 and stabilizing around 170K in Q4.
KEGS and NON-ALCOHOL remain near zero — their volumes are negligible relative to BEER.

**Chart 2 — Forecast by Supplier (Top 3)**
Crown Imports leads volume for most of the year, peaking at ~82K units in June.
Miller Brewing follows a similar seasonal curve, peaking around the same period.
Anheuser Busch shows the sharpest seasonal swing — from ~51K in January to ~74K in
May, then dropping sharply to ~50K in November. The most volatile of the three.
All three suppliers show the same summer peak pattern, confirming that seasonality
is a market-wide effect rather than supplier-specific behavior.

**Key finding:** The model projects a summer demand peak for all top suppliers —
Q2 is the highest demand period. Inventory planning should prioritize BEER stockpiling
in Q1 to meet the April–June peak, with gradual reduction through Q3 and Q4.

> **Note:** Lag features are held constant at the last known values from 2020.
> This is a baseline forecast — in production, lag features would update monthly
> with actual sales data as it becomes available.

In [0]:
# DEMAND FORECAST — 2025

import mlflow.pyfunc
import pickle

# Get the latest version number automatically from Unity Catalog
client         = mlflow.tracking.MlflowClient()
model_versions = client.search_model_versions("name='workspace.default.lightgbm_sales_forecaster'")
latest_version = max(model_versions, key=lambda v: int(v.version)).version

print(f"✓ Loaded model version {latest_version}")

# Load the model from Unity Catalog
loaded_model = mlflow.pyfunc.load_model(f"models:/workspace.default.lightgbm_sales_forecaster/{latest_version}")

# Load encoding artifacts from the latest model version
client         = mlflow.tracking.MlflowClient()
model_versions = client.search_model_versions("name='workspace.default.lightgbm_sales_forecaster'")
run_id         = model_versions[0].run_id
tmp_path       = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="encoders")

with open(f"{tmp_path}/dummy_cols.pkl", "rb") as f:
    dummy_cols_loaded = pickle.load(f)

with open(f"{tmp_path}/tier_order.pkl", "rb") as f:
    tier_order_loaded = pickle.load(f)

print("✓ Model and encoders loaded")

# Calculate top3 suppliers automatically from the data
top3_suppliers = (
    ml_df.groupby("supplier")["lag_1_sales"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)

print(f"✓ Top 3 suppliers: {top3_suppliers}")

# Get last known feature values per supplier + item_type combination
last_known = (
    ml_df.sort_values(["year", "month"])
    .groupby(["item_type", "supplier", "supplier_tier"])
    .last()
    .reset_index()
)

# Filter to top3 suppliers only
forecast_base = last_known[last_known["supplier"].isin(top3_suppliers)].copy()

# Feature columns — derived from artifacts, nothing hardcoded
feature_cols = [
    *dummy_cols_loaded,
    "supplier_tier_enc",
    "lag_1_sales",
    "lag_2_sales",
    "rolling_3m_avg",
    "warehouse_ratio",
    "transaction_count",
    "month_sin",
    "month_cos",
    "year",
]

# Generate one forecast record per month per supplier+item_type combination
forecasts = []

for month in range(1, 13):
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)

    for _, row in forecast_base.iterrows():
        forecasts.append({
            "item_type"        : row["item_type"],
            "supplier"         : row["supplier"],
            "supplier_tier"    : row["supplier_tier"],
            "lag_1_sales"      : row["lag_1_sales"],
            "lag_2_sales"      : row["lag_2_sales"],
            "rolling_3m_avg"   : row["rolling_3m_avg"],
            "warehouse_ratio"  : row["warehouse_ratio"],
            "transaction_count": row["transaction_count"],
            "month_sin"        : month_sin,
            "month_cos"        : month_cos,
            "year"             : 2025,
            "month"            : month
        })

forecast_df = pd.DataFrame(forecasts)

# Apply supplier_tier encoding using loaded artifact
forecast_df["supplier_tier_enc"] = forecast_df["supplier_tier"].map(tier_order_loaded)

# Apply one-hot encoding and align columns to training schema
dummies_forecast = pd.get_dummies(forecast_df["item_type"], prefix="type")
dummies_forecast = dummies_forecast.reindex(columns=dummy_cols_loaded, fill_value=0)
forecast_df      = pd.concat([forecast_df, dummies_forecast], axis=1)

# Generate predictions
X_forecast = forecast_df[feature_cols].values.astype(object)
forecast_df["predicted_sales"] = loaded_model.predict(X_forecast)

print(f"✓ Forecast generated — {len(forecast_df):,} records")
print(forecast_df[["supplier", "item_type", "month", "predicted_sales"]].head(10))

In [0]:
# FORECAST VISUALIZATION

# Aggregate predicted sales by month and item_type
forecast_by_type = (
    forecast_df.groupby(["month", "item_type"])["predicted_sales"]
    .sum()
    .reset_index()
)

# Aggregate predicted sales by month and supplier
forecast_by_supplier = (
    forecast_df.groupby(["month", "supplier"])["predicted_sales"]
    .sum()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1 — Forecast by item_type
colors = {
    "BEER"        : "#1D9E75",
    "WINE"        : "#2D6FA4",
    "LIQUOR"      : "#D85A30",
    "KEGS"        : "#8B5CF6",
    "NON-ALCOHOL" : "#F59E0B",
    "STR_SUPPLIES": "#6B7280"
}

for item_type, group in forecast_by_type.groupby("item_type"):
    axes[0].plot(group["month"], group["predicted_sales"],
                 marker="o", linewidth=1.8,
                 label=item_type,
                 color=colors.get(item_type, "#999"))

axes[0].set_title("2025 Demand Forecast by Item Type", fontsize=12)
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Predicted Sales (units)")
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                          "Jul","Aug","Sep","Oct","Nov","Dec"], fontsize=8)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
axes[0].spines[["top", "right"]].set_visible(False)
axes[0].legend(loc="upper left", frameon=False, fontsize=8)

# Chart 2 — Forecast by supplier
supplier_colors = {
    "CROWN IMPORTS"        : "#1D9E75",
    "ANHEUSER BUSCH INC"   : "#2D6FA4",
    "MILLER BREWING COMPANY": "#D85A30"
}

for supplier, group in forecast_by_supplier.groupby("supplier"):
    axes[1].plot(group["month"], group["predicted_sales"],
                 marker="o", linewidth=1.8,
                 label=supplier,
                 color=supplier_colors.get(supplier, "#999"))

axes[1].set_title("2025 Demand Forecast by Supplier (Top 3)", fontsize=12)
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Predicted Sales (units)")
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                          "Jul","Aug","Sep","Oct","Nov","Dec"], fontsize=8)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].legend(loc="upper left", frameon=False, fontsize=8)

plt.suptitle("LightGBM — 2025 Demand Forecast (Top 3 Suppliers)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Summary

This notebook completes the Warehouse & Retail Sales ML Pipeline by translating five notebooks of data processing and modeling into clear business visualizations.

### What was built

**Sales Overview**
BEER dominates the market with 7.67M units — 62.8% of total volume. The market is highly concentrated: Crown Imports, Anheuser Busch, and Miller Brewing control the majority of warehouse sales. From position 4 onwards volume drops sharply — no other supplier comes close to the top 3.

**Sales Trends (2017–2020)**
BEER shows consistent summer seasonality — demand peaks every year between April and June, then drops through Q4. The COVID-19 demand shock is visible in early 2020 as an unusual spike followed by a sharp collapse. WINE is the second most visible category at ~100K units monthly, stable with a slight upward trend.

**Model Performance by Segment**
LightGBM explains 96.81% of sales variance across all segments. LIQUOR (R² 95.25%) and WINE (R² 89.87%) are production-ready. BEER has the highest dollar error ($909.29) but the underlying pattern is well captured — large errors are COVID-driven, not systematic. KEGS has negative R² (-0.0257) and requires additional features to forecast reliably.

**2025 Demand Forecast**
The model projects a summer peak for all top suppliers in April–June 2025, consistent with historical seasonality. Crown Imports leads volume throughout the year peaking at ~82K units in June. All three top suppliers show the same seasonal curve — confirming that seasonality is a market-wide effect. Inventory planning should prioritize BEER stockpiling in Q1 to meet the April–June peak.

### Pipeline overview

```
Raw CSV (307,645 rows)
        │
        ▼
01 Bronze    →  Delta Table: raw ingestion, zero transformations
        │
        ▼
02 Silver    →  Delta Table: cleaned, typed, enriched — 0 nulls
        │
        ▼
03 EDA       →  Key findings: market concentration, channel behavior, seasonality
        │
        ▼
04 Gold      →  Delta Tables: business metrics + ML-ready features
        │
        ▼
05 ML Model  →  LightGBM registered in MLflow · R² 96.81% · MAE $279.45
        │
        ▼
06 Dashboard →  Business visualizations + 2025 demand forecast
```

### Author
**Santiago López Blanco**  
Data Science Engineering Student — Universidad Fidélitas, Costa Rica
